# MODULUS Pedagogical Walkthrough (Colab)

This notebook is a comprehensive, teaching-first walkthrough of the MODULUS optimization toolkit.
It is designed for researchers and engineers who want to understand not just **how** to run the code,
but **why** the design exists and where it fits in practical experimentation.

You will learn:
1. The geometric motivation behind Hyperball updates.
2. How grouped optimization policies map to transformer-style parameter trees.
3. How LoRA gradient steering is implemented and when it is useful.
4. How to run benchmark and reporting scripts and interpret results.

This notebook prioritizes conceptual clarity and reproducible hands-on demonstrations.


## Learning Roadmap

The notebook follows a progression from fundamentals to engineering operations:

1. **Environment setup** for Colab/local usage.
2. **Synthetic training baseline** with standard Optax optimizer.
3. **Hyperball-enabled training** and comparison against baseline.
4. **Grouped Hyperball policy** using path-based labels.
5. **LoRA gradient steering** intuition and concrete gradient manipulation.
6. **Benchmark/report pipeline** usage and interpretation.
7. **Exercises and extension prompts** for deeper work.

If you are short on time, run sections in order and stop after the benchmark report.
If you are onboarding a teammate, use the markdown explanations as lecture notes.

**Important:** Run the setup/bootstrap cell before executing any import cells.
This notebook now auto-detects Colab TPU runtimes and installs TPU-enabled JAX.
If TPU install path is used, run this setup cell before any JAX import; restart runtime if needed.


In [ ]:
# Self-sufficient Colab bootstrap (CPU/GPU/TPU)
# This cell makes the notebook runnable from a fresh Colab runtime.

import os
import sys
import subprocess
import importlib.util
from pathlib import Path

REPO_URL = "https://github.com/fyremael/MODULUS.git"
REPO_NAME = "MODULUS"


def run(cmd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


def in_modulus_repo(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modulus").exists()


def is_colab_runtime() -> bool:
    return importlib.util.find_spec("google.colab") is not None


def detect_colab_accelerator() -> str:
    # Colab TPU env keys vary across runtime generations; probe multiple hints.
    if os.environ.get("COLAB_GPU"):
        return "gpu"
    tpu_hints = [
        os.environ.get("COLAB_TPU_ADDR"),
        os.environ.get("TPU_NAME"),
        os.environ.get("TPU_WORKER_ID"),
        os.environ.get("ACCELERATOR_TYPE"),
    ]
    if any(v for v in tpu_hints if v):
        return "tpu"
    if Path("/dev/accel0").exists():
        return "tpu"
    return "cpu"


def uninstall_if_present(packages):
    run([sys.executable, "-m", "pip", "uninstall", "-y", *packages])


def repair_numpy_stack():
    # No-op: kept for backward compatibility with older notebook revisions.
    return


def verify_jax_runtime_backend():
    run([
        sys.executable,
        "-c",
        "import jax; print('jax_version', jax.__version__); print('jax_backend', jax.default_backend()); print('jax_devices', jax.devices())",
    ])


# Important: JAX must not be imported before install selection, or version swaps are unsafe.
if "jax" in sys.modules:
    raise RuntimeError(
        "JAX was already imported in this runtime. Restart runtime and run this setup cell first."
    )

cwd = Path.cwd()

# 1) Ensure we are inside a MODULUS checkout.
if in_modulus_repo(cwd):
    repo_dir = cwd
else:
    repo_dir = cwd / REPO_NAME
    if not repo_dir.exists():
        run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    os.chdir(repo_dir)

# In Colab, refresh checkout so notebook and scripts stay in sync across runs.
if is_colab_runtime():
    run(["git", "-C", str(repo_dir), "pull", "--ff-only"])

# 2) Runtime-specific dependency installation.
colab_accel = detect_colab_accelerator() if is_colab_runtime() else "local"
print("Detected accelerator:", colab_accel)

if colab_accel == "tpu":
    # TPU path: force-reinstall TPU JAX stack so PJRT framework/plugin versions match.
    print("TPU env:", {k: os.environ.get(k) for k in ("COLAB_TPU_ADDR", "TPU_NAME", "ACCELERATOR_TYPE")})
    uninstall_if_present([
        "jax",
        "jaxlib",
        "jax-tpu-client",
        "libtpu",
        "libtpu-nightly",
    ])
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "jax[tpu]",
        "-f",
        "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
    ])
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "optax",
        "matplotlib==3.10.1",
    ])
    # Install MODULUS package itself without forcing pinned core deps.
    uninstall_if_present(["modulus-hyperball"])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
    run([sys.executable, "-m", "pip", "show", "jax", "jaxlib"])
    try:
        verify_jax_runtime_backend()
    except Exception as exc:
        raise RuntimeError(
            "TPU backend verification failed after reinstall. Use Runtime -> Factory reset runtime, "
            "run setup first, and do not import jax before setup."
        ) from exc
else:
    # CPU/GPU path: standard extras install from project pins.
    if is_colab_runtime() and colab_accel == "cpu":
        print("Colab CPU runtime detected; using pinned CPU/GPU dependency set.")
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[report]"])

# 3) Sanity check import.
if importlib.util.find_spec("modulus") is None:
    raise RuntimeError("MODULUS import failed after installation.")

print(f"Bootstrap complete. Working directory: {Path.cwd()}")
print("Detected accelerator:", colab_accel)
print("If you changed JAX versions in-session, restart runtime before continuing.")



In [ ]:
import math
import time
from collections import Counter

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

from modulus.optim.groups import make_grouped_hyperball_tx, make_llm_default_labels
from modulus.optim.hyperball import hyperball
from modulus.optim.masks import default_llm_hyperball_mask
from modulus.peft.lora import apply_lora_grad_hook

print("JAX version:", jax.__version__)
try:
    print("JAX backend:", jax.default_backend())
    print("JAX devices:", jax.devices())
except Exception as exc:
    raise RuntimeError(
        "JAX backend initialization failed. In Colab TPU runtimes: Runtime -> Factory reset runtime, "
        "run setup cell first, then rerun imports."
    ) from exc



## Section 1: Geometric Motivation

Most optimizers treat updates as unconstrained vectors in parameter space.
Hyperball introduces a geometric view:

- We can decompose an update into **tangent** and **radial** components relative to current weights.
- We can control the step using an explicit **target angle**.
- In sphere mode, we retract back to a fixed radius after the update.

Why this matters pedagogically:
1. It makes optimizer behavior interpretable in terms of rotation vs magnitude drift.
2. It gives direct knobs for ablation: projection on/off, angle schedule, group-wise policies.
3. It enables a cleaner research vocabulary when discussing stability and conditioning.

In this notebook, we use a small synthetic model so these ideas are visible quickly.


In [ ]:
# Synthetic model with a transformer-like parameter tree.
# This keeps grouped labels/masks meaningful while remaining lightweight.

def make_params(key, width=64, rank=8):
    k1, k2, k3, k4 = jax.random.split(key, 4)
    scale = 1.0 / jnp.sqrt(float(width))
    return {
        "attn": {"kernel": jax.random.normal(k1, (width, width)) * scale},
        "mlp": {
            "kernel": jax.random.normal(k2, (width, width)) * scale,
            "adapter": {
                "lora_A": jax.random.normal(k3, (width, rank)) * 0.01,
                "lora_B": jnp.zeros((rank, width), dtype=jnp.float32),
            },
        },
        "embed": {"embedding": jax.random.normal(k4, (128, width)) * 0.01},
        "LayerNorm_0": {"scale": jnp.ones((width,), dtype=jnp.float32)},
    }


def model_forward(params, x):
    attn = x @ params["attn"]["kernel"]
    mlp = x @ params["mlp"]["kernel"]
    lora = (x @ params["mlp"]["adapter"]["lora_A"]) @ params["mlp"]["adapter"]["lora_B"]
    return attn + mlp + lora


def mse_loss(params, x, y):
    pred = model_forward(params, x)
    return jnp.mean((pred - y) ** 2)


seed = 0
width = 64
batch_size = 64
steps = 80

rng = jax.random.PRNGKey(seed)
k_params, k_teacher, k_batch = jax.random.split(rng, 3)

params_init = make_params(k_params, width=width, rank=8)
teacher = jax.random.normal(k_teacher, (width, width)) * (1.0 / jnp.sqrt(float(width)))
x_all = jax.random.normal(k_batch, (steps, batch_size, width))
y_all = jnp.einsum("tbd,df->tbf", x_all, teacher)

print("Synthetic setup ready.")


## Section 2: Baseline Training (No Hyperball)

Before adding geometric constraints, establish baseline behavior.
This is essential pedagogy: you need a control condition for any claimed gain.

We train with `adamw` only and record:
- loss trajectory
- average step time

Then we will run the same setup with Hyperball and compare directly.


In [ ]:
def run_baseline(params, x_all, y_all, lr=1e-3, wd=1e-2):
    tx = optax.adamw(lr, weight_decay=wd)
    opt_state = tx.init(params)

    @jax.jit
    def step_fn(p, s, x, y):
        loss_val, grads = jax.value_and_grad(mse_loss)(p, x, y)
        updates, s2 = tx.update(grads, s, p)
        p2 = optax.apply_updates(p, updates)
        return p2, s2, loss_val

    losses = []
    times_ms = []
    p = params
    s = opt_state
    for i in range(x_all.shape[0]):
        t0 = time.perf_counter()
        p, s, loss_val = step_fn(p, s, x_all[i], y_all[i])
        jax.block_until_ready(loss_val)
        times_ms.append((time.perf_counter() - t0) * 1000.0)
        losses.append(float(loss_val))
    return p, losses, times_ms


params_b, losses_b, times_b = run_baseline(params_init, x_all, y_all)
print(f"baseline final loss: {losses_b[-1]:.6f}, avg step ms: {sum(times_b)/len(times_b):.4f}")


## Section 3: Hyperball Training

Now we wrap the same base optimizer with Hyperball.

Conceptually, this does three things:
1. Optional tangent projection removes radial drift components.
2. Optional target angle rescales update geometry to a controlled rotational step.
3. Sphere retraction maintains norm constraints per selected granularity.

For a fair comparison, all non-Hyperball settings (data, model, base optimizer) remain fixed.


In [ ]:
def run_hyperball(params, x_all, y_all, lr=1e-3, wd=1e-2, angle=0.04):
    base = optax.adamw(lr, weight_decay=wd)
    mask_fn = default_llm_hyperball_mask(
        include_embeddings=False,
        exclude_lora=True,
        exclude_1d=True,
    )
    tx = hyperball(
        base,
        radius=1.0,
        mode="sphere",
        proj_tangent=True,
        granularity="row",
        target_angle=angle,
        mask=mask_fn,
        emit_metrics=True,
    )
    opt_state = tx.init(params)

    @jax.jit
    def step_fn(p, s, x, y):
        loss_val, grads = jax.value_and_grad(mse_loss)(p, x, y)
        updates, s2 = tx.update(grads, s, p)
        p2 = optax.apply_updates(p, updates)
        return p2, s2, loss_val

    losses = []
    times_ms = []
    p = params
    s = opt_state
    for i in range(x_all.shape[0]):
        t0 = time.perf_counter()
        p, s, loss_val = step_fn(p, s, x_all[i], y_all[i])
        jax.block_until_ready(loss_val)
        times_ms.append((time.perf_counter() - t0) * 1000.0)
        losses.append(float(loss_val))
    return p, losses, times_ms, s


params_h, losses_h, times_h, opt_state_h = run_hyperball(params_init, x_all, y_all)
print(f"hyperball final loss: {losses_h[-1]:.6f}, avg step ms: {sum(times_h)/len(times_h):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses_b, label="baseline")
plt.plot(losses_h, label="hyperball")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Baseline vs Hyperball")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Section 4: Grouped Hyperball Policies

Grouped policies let you assign different optimizer behavior to different parameter regions.
In LLM-like settings, attention and MLP blocks may benefit from different angle schedules.

Pedagogical point:
- Grouped transforms are only as good as your labels.
- Always inspect label coverage before trusting results.


In [ ]:
labels_fn = make_llm_default_labels()
labels = labels_fn(params_init)

def flatten(tree):
    out = []
    def rec(node):
        if isinstance(node, dict):
            for v in node.values():
                rec(v)
        else:
            out.append(node)
    rec(tree)
    return out

label_counts = Counter(str(x) for x in flatten(labels))
print("Label distribution:")
for k in sorted(label_counts):
    print(f"- {k}: {label_counts[k]}")

base = optax.adamw(1e-3, weight_decay=1e-2)
mask_fn = default_llm_hyperball_mask(include_embeddings=False, exclude_lora=True, exclude_1d=True)
hb_common = dict(radius=1.0, mode="sphere", proj_tangent=True, granularity="row", mask=mask_fn)

tx_builder = make_grouped_hyperball_tx(
    base_by_group={k: base for k in ["attn", "mlp", "other", "embed", "norm", "bias"]},
    hyperball_kwargs_by_group={
        "attn": dict(**hb_common, target_angle=0.03),
        "mlp": dict(**hb_common, target_angle=0.05),
        "other": {},
        "embed": {},
        "norm": {},
        "bias": {},
    },
    labels_fn=labels_fn,
    default_group="other",
)
tx_grouped = tx_builder(params_init)
st = tx_grouped.init(params_init)
loss_val, grads = jax.value_and_grad(mse_loss)(params_init, x_all[0], y_all[0])
updates, st2 = tx_grouped.update(grads, st, params_init)
print("Grouped tx update keys:", list(updates.keys()))
print("Single-step grouped loss:", float(loss_val))


## Section 5: LoRA Gradient Steering

The LoRA hook in MODULUS applies an orthogonalization-style transformation to LoRA gradients.
Intuition:
- It removes update components aligned with existing low-rank factor subspaces.
- This can reduce redundant scaling directions and improve conditioning in some regimes.

Important practice note:
- Treat this as an ablation knob, not a guaranteed universal improvement.
- Evaluate with your benchmark matrix, not single-run intuition.


In [ ]:
# Build a tiny param/grads tree with LoRA slots and show hook effect on gradient norms.

p_demo = {
    "block": {
        "kernel": jnp.ones((8, 8), dtype=jnp.float32),
        "lora_A": jax.random.normal(jax.random.PRNGKey(1), (8, 4)),
        "lora_B": jax.random.normal(jax.random.PRNGKey(2), (4, 8)),
    }
}
g_demo = {
    "block": {
        "kernel": jnp.ones((8, 8), dtype=jnp.float32),
        "lora_A": jax.random.normal(jax.random.PRNGKey(3), (8, 4)),
        "lora_B": jax.random.normal(jax.random.PRNGKey(4), (4, 8)),
    }
}

g_hook = apply_lora_grad_hook(p_demo, g_demo, a_name="lora_A", b_name="lora_B", eps=1e-6)

na0 = float(jnp.linalg.norm(g_demo["block"]["lora_A"]))
nb0 = float(jnp.linalg.norm(g_demo["block"]["lora_B"]))
na1 = float(jnp.linalg.norm(g_hook["block"]["lora_A"]))
nb1 = float(jnp.linalg.norm(g_hook["block"]["lora_B"]))

print(f"A grad norm before/after: {na0:.6f} -> {na1:.6f}")
print(f"B grad norm before/after: {nb0:.6f} -> {nb1:.6f}")


## Section 6: Benchmark and Reporting Pipeline

MODULUS includes script-level tooling so experiments produce consistent artifacts.

You can run:
1. `scripts/run_benchmarks.py` to generate CSV metrics.
2. `scripts/build_benchmark_report.py` to synthesize a human-readable report.

The benchmark now uses a more realistic stressor:
- Multi-layer transformer-style sequence model (attention + MLP + residual + layer norm).
- Distillation objective (teacher/student KL) mixed with next-token CE.
- Mid-run token distribution shift plus rare-token injection.
- Periodic held-out evaluation and throughput/norm telemetry.
- In-notebook execution path to avoid TPU subprocess re-entrancy issues.
- Optional real-world text streaming from Hugging Face rows API (minimal deps).
- Persistent Hugging Face cache wiring (`HF_HOME`, datasets cache, and rows-page cache) for faster reruns.
- Hardware-aware safety guards auto-reduce `batch_size` and token pool pressure when device/RAM limits are tighter than requested.
- Memory-safe logging path: step rows stream to CSV with configurable sampling interval.
- Colab TPU v6e1 ablation profiles: quick smoke, balanced 30-minute, and full 1B-token run.
- Model sizing set to NanoGPT-class scale (~136M parameters in this benchmark architecture).
- Default profile runs all five configs for side-by-side engineering comparison.

This is critical for team pedagogy:
- It shifts discussion from anecdotes to evidence.
- It keeps comparisons reproducible across contributors.


In [ ]:
import importlib.util
import csv
import os
import sys
from pathlib import Path


def load_module(module_path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, str(module_path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Failed to load module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


bench_dir = Path("artifacts/benchmarks/notebook_demo")

# Real-world streaming dataset options:
# - "HuggingFaceFW/fineweb" (default below, using sample-10BT)
# - "JeanKaddour/minipile"
dataset_name = "HuggingFaceFW/fineweb"
dataset_config = "sample-10BT"
dataset_train_split = "train"
dataset_eval_split = "train"
dataset_text_keys = "text,content,document"
dataset_token_env = "HF_TOKEN"
data_source = "hf_http"  # switch to "hf_stream" if datasets streaming is stable in runtime
dataset_cache_root = Path("/content/cache/modulus_hf")
hf_home = dataset_cache_root / "huggingface"
rows_cache_dir = dataset_cache_root / "rows_cache"
hf_home.mkdir(parents=True, exist_ok=True)
rows_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(hf_home))
os.environ.setdefault("HF_DATASETS_CACHE", str(hf_home / "datasets"))
os.environ.setdefault("HF_HUB_CACHE", str(hf_home / "hub"))
ablation_profile = "balanced_30min"

try:
    from google.colab import userdata  # type: ignore
    token_from_secret = userdata.get("HF_TOKEN")
    if token_from_secret:
        os.environ[dataset_token_env] = token_from_secret
except Exception:
    pass

# TPU-safe single-process execution: avoid subprocess on TPU runtimes.
bench_mod = load_module(Path("scripts/run_benchmarks.py"), "modulus_run_benchmarks")
bench_args = bench_mod.parse_args([])

base_args = {
    "warmup_steps": 8,
    "configs": "baseline,lora_hook_only,hyperball_ungrouped,hyperball_grouped,hyperball_grouped_lora",
    "hardware_aware": True,
    "max_tokens_per_step": 8192,
    "auto_token_pool_by_host_ram": True,
    "host_ram_token_pool_fraction": 0.20,
    "batch_size": 12,
    "seq_len": 1024,
    "width": 768,
    "num_layers": 12,
    "num_heads": 12,
    "mlp_mult": 4,
    "vocab_size": 32768,
    "lora_rank": 8,
    "grad_accum_steps": 3,
    "lr": 3e-4,
    "lr_schedule": "warmup_cosine",
    "lr_warmup_steps": 500,
    "lr_min_ratio": 0.10,
    "step_record_interval": 20,
    "out_dir": str(bench_dir),
}

profile_overrides = {
    "quick_smoke": {
        "steps": 120,
        "max_steps": 800,
        "target_runtime_minutes": 0.0,
        "target_train_tokens": 8_000_000,
        "token_pool_batches": 192,
        "eval_interval": 40,
        "eval_batches": 2,
        "log_interval": 20,
        "step_record_interval": 10,
        "lr_warmup_steps": 80,
        "lr_total_steps": 800,
    },
    "balanced_30min": {
        "steps": 1200,
        "max_steps": 12000,
        "target_runtime_minutes": 30.0,
        "target_train_tokens": None,
        "token_pool_batches": 256,
        "eval_interval": 100,
        "eval_batches": 2,
        "log_interval": 25,
        "step_record_interval": 20,
        "lr_warmup_steps": 600,
        "lr_total_steps": 12000,
    },
    "full_1b": {
        "steps": 1000,
        "max_steps": 100000,
        "target_runtime_minutes": 0.0,
        "target_train_tokens": 1_000_000_000,
        "token_pool_batches": 512,
        "eval_interval": 250,
        "eval_batches": 2,
        "log_interval": 50,
        "step_record_interval": 25,
        "lr_warmup_steps": 3000,
        "lr_total_steps": 100000,
    },
}

if ablation_profile not in profile_overrides:
    raise ValueError(
        f"Unknown ablation_profile={ablation_profile!r}. "
        f"Choose from {sorted(profile_overrides.keys())}."
    )

selected_args = dict(base_args)
selected_args.update(profile_overrides[ablation_profile])
for k, v in selected_args.items():
    setattr(bench_args, k, v)

def estimate_param_count(width, num_layers, mlp_mult, vocab_size, seq_len, lora_rank):
    mlp_hidden = width * mlp_mult
    embed = vocab_size * width
    pos = seq_len * width
    head = width * vocab_size
    final_norm = width
    per_layer = (
        width * (3 * width)
        + width * width
        + width * mlp_hidden
        + mlp_hidden * width
        + (2 * width * lora_rank)
        + (2 * width)
    )
    return embed + pos + head + final_norm + (num_layers * per_layer)

param_est = estimate_param_count(
    width=bench_args.width,
    num_layers=bench_args.num_layers,
    mlp_mult=bench_args.mlp_mult,
    vocab_size=bench_args.vocab_size,
    seq_len=bench_args.seq_len,
    lora_rank=bench_args.lora_rank,
)
print(f"Approx model size: {param_est / 1_000_000:.1f}M params")
if not (120_000_000 <= param_est <= 160_000_000):
    print("WARNING: model is outside the NanoGPT-class (~140M) band.")

if not hasattr(bench_args, "data_source"):
    raise RuntimeError(
        "Outdated scripts/run_benchmarks.py detected in this Colab runtime. "
        "Re-run the setup cell (it now does git pull --ff-only), then retry this section."
    )

bench_args.data_source = data_source
bench_args.dataset_name = dataset_name
bench_args.dataset_train_split = dataset_train_split
bench_args.dataset_eval_split = dataset_eval_split
bench_args.dataset_text_keys = dataset_text_keys
bench_args.dataset_shuffle_buffer = 20000
bench_args.dataset_max_doc_tokens = 2048
bench_args.dataset_eval_fallback_to_train = True
bench_args.dataset_config = dataset_config
bench_args.dataset_rows_page_size = 100
bench_args.dataset_http_max_retries = 30
bench_args.dataset_http_min_interval_sec = 0.15
bench_args.dataset_http_token_env = dataset_token_env
bench_args.dataset_http_cache_dir = str(rows_cache_dir)
bench_args.dataset_http_cache_read = True
bench_args.dataset_http_cache_write = True

n_cfg = len([c for c in bench_args.configs.split(",") if c.strip()])
print(
    f"Benchmark profile={ablation_profile} | configs={n_cfg} | "
    f"lr={bench_args.lr} ({bench_args.lr_schedule})"
)
if bench_args.target_train_tokens is not None:
    total_target_tokens = n_cfg * int(bench_args.target_train_tokens)
    print(
        "Token budget: "
        f"per_config={bench_args.target_train_tokens:,} | "
        f"aggregate={total_target_tokens:,}"
    )
else:
    print(
        "Runtime budget: "
        f"{bench_args.target_runtime_minutes:.1f} minutes minimum per run loop"
    )

bench_mod.run(bench_args)

report_mod = load_module(Path("scripts/build_benchmark_report.py"), "modulus_build_report")
report_args = report_mod.parse_args(["--run-dir", str(bench_dir), "--with-plots"])
report_mod.run(report_args)

summary_path = bench_dir / "benchmark_summary.csv"
rows = list(csv.DictReader(summary_path.open("r", encoding="utf-8")))
rows.sort(key=lambda r: float(r["final_eval_loss"]))
baseline_row = next((r for r in rows if r["config"] == "baseline"), None)
baseline_eval = float(baseline_row["final_eval_loss"]) if baseline_row else float("nan")
baseline_speed = float(baseline_row["avg_tokens_per_s"]) if baseline_row else float("nan")

print("\nAblation snapshot (sorted by final_eval_loss):")
for r in rows:
    eval_loss = float(r["final_eval_loss"])
    speed = float(r["avg_tokens_per_s"])
    loss_delta = eval_loss - baseline_eval if baseline_row else float("nan")
    speed_ratio = speed / baseline_speed if baseline_row and baseline_speed > 0 else float("nan")
    print(
        f"- {r['config']}: final_eval_loss={eval_loss:.6f} "
        f"(delta_vs_baseline={loss_delta:+.6f}), avg_tokens_per_s={speed:.2f} "
        f"(speed_ratio_vs_baseline={speed_ratio:.3f})"
    )

print("Benchmark and report generation complete.")
print("Open:", str(bench_dir / "benchmark_report.md"))


## Exercises (Recommended)

1. **Angle sensitivity study**
- Re-run Hyperball with target angles `[0.01, 0.02, 0.04, 0.08]`.
- Plot convergence speed vs stability.

2. **Mask policy ablation**
- Toggle `include_embeddings` and `exclude_lora`.
- Observe changes in loss and runtime.

3. **Grouped policy exploration**
- Keep attention angle fixed and sweep MLP angle.
- Compare to ungrouped Hyperball.

4. **Pedagogical extension**
- Add your own markdown section explaining one experiment result in plain language.
- This is excellent onboarding practice for junior engineers.

5. **Production adaptation**
- Replace synthetic params with your model tree.
- Validate labeling/masking using `scripts/validate_integration_tree.py` ideas.


## Closing Notes

You now have a practical understanding of the full MODULUS loop:
concept -> implementation -> ablation -> report -> engineering gate.

Suggested next actions:
1. Convert one of your real training jobs to grouped Hyperball as a controlled experiment.
2. Standardize your benchmark configuration and report interpretation rubric.
3. Keep docs and API reference synchronized via CI freshness checks.

If you use this notebook for team onboarding, consider requiring each new engineer to submit
one benchmark variation with a short written interpretation as part of ramp-up.
